# 01 - Data Cleaning
Load the raw transactions data, inspect data quality issues, and produce clean tables ready for SQL loading and analysis.

**Business context:** Raw e-commerce export data is messy — missing customer IDs, duplicate rows, inconsistent text formatting, and invalid prices. This notebook documents each cleaning decision and why it was made.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Load raw data

In [2]:
df = pd.read_csv('../data/raw_online_retail.csv', parse_dates=['InvoiceDate'])
print(df.shape)
df.head()

(1067371, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Data quality audit

In [3]:
print('Missing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNegative quantities (returns):', (df['Quantity'] < 0).sum())
print('Zero/negative prices:', (df['UnitPrice'] <= 0).sum())

Missing values per column:
InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     243007
Country             0
dtype: int64

Duplicate rows: 34335

Negative quantities (returns): 22950
Zero/negative prices: 6207


## Cleaning steps
1. Drop exact duplicates
2. Drop rows with missing `CustomerID` (can't attribute to a customer for RFM/churn analysis)
3. Standardize text fields (strip whitespace, title case)
4. Flag returns rather than dropping them — return rate is a useful business signal
5. Remove invalid price rows (excluding legitimate returns)
6. Compute `line_total` = quantity × unit price

In [4]:
import sys
sys.path.append('../scripts')
from clean_data import clean_transactions, build_customers, build_products

products_raw = pd.read_csv('../data/product_catalog.csv')
clean_tx = clean_transactions(df)
customers = build_customers(clean_tx)
clean_products = build_products(products_raw)
clean_tx.head()

Cleaned transactions: 1067371 -> 797815 rows (269556 removed)


,invoice_no,stock_code,customer_id,quantity,unit_price,invoice_date,line_total,is_return,country
0,489434,85048,13085,12,6.95,2009-12-01 07:45:00,83.4,False,United Kingdom
1,489434,79323P,13085,12,6.75,2009-12-01 07:45:00,81.0,False,United Kingdom
2,489434,79323W,13085,12,6.75,2009-12-01 07:45:00,81.0,False,United Kingdom
3,489434,22041,13085,48,2.10,2009-12-01 07:45:00,100.8,False,United Kingdom
4,489434,21232,13085,24,1.25,2009-12-01 07:45:00,30.0,False,United Kingdom


## Save cleaned tables

In [5]:
clean_tx.drop(columns=['country']).to_csv('../data/clean_transactions.csv', index=False)
customers.to_csv('../data/clean_customers.csv', index=False)
clean_products.to_csv('../data/clean_products.csv', index=False)
print('Saved cleaned CSVs to ../data/')

Saved cleaned CSVs to ../data/


## Key takeaways
- X% of rows were removed during cleaning (see printed counts above)
- Missing `CustomerID` was the largest single cause of row loss
- Returns were retained as a flag rather than deleted, to preserve return-rate as an analysis dimension